In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import torchvision 
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms
from torch.utils.data import DataLoader,TensorDataset

In [3]:
# Scale: (0,1)
# Normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_dataset = CIFAR10(root = "./data",train=True,download=True,transform=transform)
test_dataset = CIFAR10(root = "./data",train=False,download=True,transform=transform)

In [4]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 64,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size = 64,
    shuffle=True
)

In [5]:
class Classifier(nn.Module):

    def __init__(self):
        super(Classifier,self).__init__()

        self.conv_layers = nn.Sequential(

            # Layer 1

            nn.Conv2d(
                in_channels=3,
                out_channels=32, # Generally powers of 2
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),
            
            # Layer 2

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Layer 3

            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        # Fully Connected layers

        self.fc_layers = nn.Sequential(
            nn.Linear(
                in_features=4*4*128,
                out_features=256
            ),
            nn.ReLU(),

            nn.Linear(
                in_features=256,
                out_features=10
            )
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # Flattening

        return self.fc_layers(x)

In [6]:
model = Classifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [7]:
epochs = 10
best_val_loss = float("inf")
for epoch in range(epochs):
    time = datetime.now()
    model.train()
    epoch_training_loss = 0.0

    for images,labels in train_loader:

        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output,labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss +=loss.item()
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images,labels in test_loader:
            output = model.forward(images)
            loss = criterion(output,labels)
            val_loss+=loss.item()
    val_loss = val_loss/len(test_loader)
    if(val_loss<best_val_loss):
        best_val_loss= val_loss
        torch.save(model.state_dict(),"best_model.pt")
    time2 = datetime.now()
    print(f"Epoch: {epoch+1}/{epochs} || Loss: {epoch_training_loss/len(train_loader)} || Time taken: {time2-time}") 

Epoch: 1/10 || Loss: 1.3831304051077273 || Time taken: 0:00:24.718894
Epoch: 2/10 || Loss: 0.9355701352171886 || Time taken: 0:00:26.457104
Epoch: 3/10 || Loss: 0.7481061532293134 || Time taken: 0:00:25.428465
Epoch: 4/10 || Loss: 0.6183424037512001 || Time taken: 0:00:25.534096
Epoch: 5/10 || Loss: 0.5178688660149684 || Time taken: 0:00:26.837118
Epoch: 6/10 || Loss: 0.4162819930697646 || Time taken: 0:00:25.484773
Epoch: 7/10 || Loss: 0.33242927090553065 || Time taken: 0:00:25.709750
Epoch: 8/10 || Loss: 0.26199328895572505 || Time taken: 0:00:33.286980
Epoch: 9/10 || Loss: 0.2018811034415003 || Time taken: 0:00:25.657525
Epoch: 10/10 || Loss: 0.157321746187175 || Time taken: 0:00:26.114388


In [8]:
correct = 0
total = 0
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

with torch.no_grad():
    for images,labels in test_loader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)
        correct +=(predicted==labels).sum().item()
        total += labels.size(0)

In [ ]:
with torch.no_grad():
    for images,labels in test_dataset:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)
        correct +=(predicted==labels).sum().item()
        total += labels.size(0)
        

Accuracy: 0.755
